### Librerías a utilizar
---

In [1]:
import pickle
import pandas as pd
import numpy as np
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.feature_extraction import  DictVectorizer
from sklearn.preprocessing import RobustScaler
from dotenv import load_dotenv
import statsmodels.api as sm
import math
import optuna
import pathlib
from optuna.samplers import TPESampler
from sklearn.preprocessing import StandardScaler
from mlflow.models.signature import infer_signature
import mlflow
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss, accuracy_score, f1_score
from sklearn.ensemble import RandomForestClassifier
import xgboost as xgb
from mlflow import MlflowClient
from datetime import datetime
import mlflow.pyfunc as mlflow_pyfunc
from sklearn.feature_selection import mutual_info_classif
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Cargar las credenciales
---

In [2]:
load_dotenv(override=True)  # Carga las variables del archivo .env
EXPERIMENT_NAME = "/Users/monica.ibarra@iteso.mx/project1-experiment" 

mlflow.set_tracking_uri("databricks")
experiment = mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

Se cargan las credenciasles necesarias para conectarse a los servicios de MLflow y el almacenamiento en la nube.

### Preprocessing
___

Para el prepocesamiento de los datos se hacen las transformaciones y limpieza necesarias que se determinaron en la estapa de análisis exploratorio de datos (EDA) y de data wrangling.

A continuación, una explicación breve del preprocesamiento realizado:

1. *Eliminación de columnas irrelevantes o redundantes*: Se eliminan variables que no aportan información útil al modelo (Health_Issues, Caffeine_mg, Sleep_Hours, Sleep_Quality)

2. *Filtrado de valores no representativos*: Se excluyen las filas con género "Other".

3. *Agrupación de los países*: Se crea una nueva variable Continent a partir del país mediante un mapeo.

4. *Codificación variables categóricas*: Stress_Level se convierte en valores numéricos:Low = 0, Medium = 1, High = 2

5. *Eliminación de columnas identificadoras*: Se eliminan columnas como ID que no aportan valor predictivo.

6. *Conversión de booleanos a enteros*: Las variables booleanas (True/False) se transforman en valores numéricos

7. *Codificación de variables categóricas con DictVectorizer*: Se transforma el dataset en formato numérico usando One-Hot Encoding 

8. *Eliminación de dummies*: De cada conjunto de variables dummies creadas por una categoría, se elimina la primera columna base para evitar multicolinealidad.

9. *Eliminación de variables constantes*: Se eliminan columnas donde todos los valores son iguales, ya que no aportan información al modelo.

10. *Eliminación de variables altamente correlacionadas*: Se calcula la matriz de correlación y se eliminan variables con correlación mayor a 0.9 para reducir redundancia.

11. *Selección de características más relevantes (Feature Selection)*: Se calcula la información mutua entre cada variable y la variable objetivo. Se conservan las características más informativas.

12. *Estandarización de variables numéricas*: Se aplica StandardScaler para centrar y escalar las variables, mejorando el desempeño de modelos sensibles a la escala.

13. *Balanceo de clases con SMOTE*:  Se genera un dataset balanceado duplicando de forma sintética las clases minoritarias, evitando sesgos del modelo hacia la clase mayoritaria.

In [3]:
def preprocessing_train(df: pd.DataFrame, n_top_features: int = 20):
    # ---- Agrupar Sleep_Quality (romper la relación perfecta) ----
    sleep_map = {
        "Poor": "Bad",
        "Fair": "Bad",
        "Good": "Good",
        "Excellent": "Good"
    }
    if "Sleep_Quality" in df.columns:
        df["Sleep_Group"] = df["Sleep_Quality"].map(sleep_map)
        df = df.drop(columns=["Sleep_Quality"], errors="ignore")

    # Eliminar columnas innecesarias y preparar datos
    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    # Mapear países a continentes
    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    if "Country" in df.columns:
        df["Continent"] = df["Country"].map(pais_a_continente)

    # Mapear nivel de estrés y género
    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})

    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    # Separar X e y antes de DictVectorizer
    y = df["Stress_Level"].values
    X_df = df.drop(columns=["Stress_Level"])

    # DictVectorizer 
    dicts = X_df.to_dict(orient="records")
    dv = DictVectorizer(sparse=False)
    X = dv.fit_transform(dicts)
    X_df_encoded = pd.DataFrame(X, columns=dv.get_feature_names_out())

    # Eliminar una dummy por cada categoría para evitar colinealidad 
    feature_names = dv.get_feature_names_out().tolist()
    groups = {}
    for f in feature_names:
        if '=' in f:
            pref = f.split('=')[0]
            groups.setdefault(pref, []).append(f)
    to_drop_dummy = []
    for pref, feats in groups.items():
        if len(feats) > 1:
            feats_sorted = sorted(feats)
            to_drop_dummy.append(feats_sorted[0])
    if to_drop_dummy:
        X_df_encoded = X_df_encoded.drop(columns=[c for c in to_drop_dummy if c in X_df_encoded.columns], errors='ignore')

    # Eliminar columnas constantes
    nunique = X_df_encoded.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X_df_encoded = X_df_encoded.drop(columns=constant_cols, errors='ignore')

    # Feature selection por correlación 
    # Eliminar variables con alta correlación (pearson) > 0.9
    corr_matrix = X_df_encoded.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop_corr = [column for column in upper.columns if any(upper[column] > 0.9)]
    if to_drop_corr:
        print("Eliminando por correlación alta:", to_drop_corr)
    X_filtered = X_df_encoded.drop(columns=to_drop_corr, errors='ignore')

    mi = mutual_info_classif(X_filtered.values, y, random_state=42)
    mi_series = pd.Series(mi, index=X_filtered.columns)

    # seleccionar top n features
    top_features = mi_series.nlargest(n_top_features).index.tolist()
    print("Top features seleccionadas:", top_features)

    top_features = [f for f in top_features if f in X_filtered.columns]
    X_selected = X_filtered[top_features]

    # Escalado
    scaler = RobustScaler()
    X_scaled = scaler.fit_transform(X_selected.values)

    # SMOTE
    smote = SMOTE(random_state=42)
    X_bal, y_bal = smote.fit_resample(X_scaled, y)

    print("Preprocessing train completado.")
    print("Distribución target (train):\n", pd.Series(y).value_counts())
    print("Distribución target (post-SMOTE):\n", pd.Series(y_bal).value_counts())

    dropped = {
        "dropped_dummy": to_drop_dummy,
        "dropped_corr": to_drop_corr
    }

    #return X_scaled, y, dv, top_features, dropped, scaler
    return X_bal, y_bal, dv, top_features, dropped, scaler

In [4]:
def preprocessing_eval(df: pd.DataFrame, dv: DictVectorizer, features, scaler: RobustScaler):
    # ---- Agrupar Sleep_Quality (misma lógica que en train) ----
    sleep_map = {
        "Poor": "Bad",
        "Fair": "Bad",
        "Good": "Good",
        "Excellent": "Good"
    }
    if "Sleep_Quality" in df.columns:
        df["Sleep_Group"] = df["Sleep_Quality"].map(sleep_map)
        df = df.drop(columns=["Sleep_Quality"], errors="ignore")

    df = df.drop(columns=['Health_Issues', 'Caffeine_mg', 'Sleep_Hours'], errors='ignore')
    df = df[df["Gender"] != "Other"]

    pais_a_continente = {
        "Canada": "America", "USA": "America", "Mexico": "America", "Brazil": "America",
        "Norway": "Europe", "Sweden": "Europe", "UK": "Europe", "Finland": "Europe",
        "Italy": "Europe", "Belgium": "Europe", "Germany": "Europe", "France": "Europe",
        "Switzerland": "Europe", "Netherlands": "Europe", "Spain": "Europe",
        "India": "Asia", "China": "Asia", "South Korea": "Asia", "Japan": "Asia",
        "Australia": "Oceania"
    }
    if "Country" in df.columns:
        df["Continent"] = df["Country"].map(pais_a_continente)

    df['Stress_Level'] = df['Stress_Level'].map({'Low': 0, 'Medium': 1, 'High': 2})
    df['Gender'] = df['Gender'].map({'Male': 0, 'Female': 1})
    df = df.drop(columns=["Country"], errors='ignore')
    if "ID" in df.columns:
        df = df.drop(columns=["ID"])

    for col in df.columns:
        if df[col].dtype == 'bool':
            df[col] = df[col].astype(int)

    dicts = df.drop(columns=["Stress_Level"]).to_dict(orient="records")
    X_encoded = dv.transform(dicts).astype(float)

    X_encoded[~np.isfinite(X_encoded)] = np.nan
    if np.isnan(X_encoded).any():
        X_encoded = np.nan_to_num(X_encoded, nan=0.0, posinf=0.0, neginf=0.0)

    feature_names = dv.get_feature_names_out()
    X_df_encoded = pd.DataFrame(X_encoded, columns=feature_names)

    nunique = X_df_encoded.nunique()
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X_df_encoded = X_df_encoded.drop(columns=constant_cols, errors='ignore')
    
    # Asegurar que todas las features estén presentes (si falta alguna, añadir con 0)
    for f in features:
        if f not in X_df_encoded.columns:
            X_df_encoded[f] = 0.0

    # Seleccionar columnas en mismo orden que 'features'
    X_filtered_df = X_df_encoded[features]

    # Transformar con scaler
    X_scaled = scaler.transform(X_filtered_df.values)

    y = df["Stress_Level"].values
    return X_scaled, y

Además de hacer la limpieza que se hizo en el preprocesamiento de entrenamiento:

- *Transforma los registros con DictVectorizer*: Convierte el dataframe sin la columna Stress_Level a una matriz numérica con las columnas que el dv aprendió en train.
- *Construye X_df_encoded*: Lo hace con los nombres de columna del dv
- *Crea un DataFrame*: Lo crea con columnas a partir de  dv.get_feature_names_out() para poder alinear columnas
- *Escala usando el scaler entrenado en train*: Para mantener la coherencia de escala entre train/val/test.
- *Devuelve X_scaled y y*: X_scaled: matriz lista para predecir con el modelo (misma cantidad y orden de features que en train).
y: vector variable objetivo


### Dividir en entrenamiento, prueba & validacion
---

In [5]:
df_raw = pd.read_csv("../data/raw/synthetic_coffee_health_10000.csv")

In [6]:
target = 'Stress_Level'

In [7]:
train_df, temp_df = train_test_split(df_raw, test_size=0.4, random_state=42, stratify=df_raw[target])
val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df[target])

In [8]:
# Preprocess train
X_train_bal, y_train_bal, dv, top_features, dropped, scaler = preprocessing_train(train_df)
#X_train_scaled, y_train, dv, top_features, dropped, scaler = preprocessing_train(train_df)

Top features seleccionadas: ['Sleep_Group=Good', 'Occupation=Student', 'Coffee_Intake', 'Physical_Activity_Hours', 'Continent=Oceania', 'BMI', 'Alcohol_Consumption', 'Age', 'Gender', 'Continent=Europe', 'Continent=Asia', 'Heart_Rate', 'Occupation=Service', 'Occupation=Other', 'Occupation=Office', 'Smoking']
Preprocessing train completado.
Distribución target (train):
 0    4095
1    1205
2     567
Name: count, dtype: int64
Distribución target (post-SMOTE):
 0    4095
2    4095
1    4095
Name: count, dtype: int64


Las predictoras que fueron seleccionadas por el método de correlación e información mutua y que se utilizarán para entrenar los modelos serán:
- 'Coffee_Intake'
- 'Occupation=Office'
- 'Occupation=Service'
- 'Alcohol_Consumption'
- 'Heart_Rate'
- 'BMI'
- 'Age'
- 'Continent=Oceania'
- 'Continent=Europe'
- 'Continent=Asia'
- 'Gender'
- 'Occupation=Other'
- 'Occupation=Student'
- 'Physical_Activity_Hours'
- 'Smoking'

In [ ]:
X_val, y_val = preprocessing_eval(val_df, dv, top_features, scaler)
X_test, y_test = preprocessing_eval(test_df, dv, top_features, scaler)

### Regresión Logística
---

El primer modelo que se entrena es una regresión logística. Se utiliza Optuna para la optimización de hiperparámetros y MLflow para el seguimiento de experimentos.

Los hiperparámetros que se optimizan son:
- *Penalty (l1, l2, elasticnet)*: Este hiperparámetro define el tipo de regularización que se aplicará al modelo.
- *C (Regularización)*: Este hiperparámetro controla la fuerza de la regularización. Un valor más pequeño indica una regularización más fuerte.
- *Class weight (None, balanced)*: Este hiperparámetro ajusta los pesos de las clases para manejar conjuntos de datos desequilibrados.
- *Solver (saga)*: Este hiperparámetro define el algoritmo a utilizar en la optimización del modelo.
- *Multi class (multinomial)*: Este hiperparámetro especifica el tipo de problema multiclase a resolver.
La función objetivo para Optuna entrena el modelo con los hiperparámetros sugeridos y evalúa su rendimiento utilizando la métrica F1 macro en el conjunto de validación. El objetivo es maximizar esta métrica.


#### Función objetivo

In [ ]:
def objective_logreg(trial: optuna.trial.Trial):
    # -----------------------------
    # Hiperparámetros a buscar
    # -----------------------------
    penalty = trial.suggest_categorical("penalty", ["l1", "l2", "elasticnet"])
    l1_ratio = None
    if penalty == "elasticnet":
        l1_ratio = trial.suggest_float("l1_ratio", 0.0, 1.0)

    C = trial.suggest_float("C", 1e-4, 1e2, log=True)
    class_weight = 'balanced'

    # -----------------------------
    # Configurar parámetros del modelo
    # -----------------------------
    params = {
        "penalty": penalty,
        "C": C,
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "solver": "saga",
        "multi_class": "multinomial",
        "max_iter": 2000,
        "random_state": 42,
        "n_jobs": -1
    }
    if penalty == "elasticnet":
        params["l1_ratio"] = l1_ratio

    # -----------------------------
    # Entrenamiento y evaluación
    # -----------------------------
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "logistic_regression")
        mlflow.log_params(params)

        # Entrenar modelo
        model = LogisticRegression(**params)
        #model.fit(X_train_scaled, y_train)
        model.fit(X_train_bal, y_train_bal)

        # Predicciones
        y_proba = model.predict_proba(X_val)
        y_pred = model.predict(X_val)

        # Métricas
        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        # Loguear métricas
        mlflow.log_metric("accuracy", val_acc)
        mlflow.log_metric("log_loss", val_logloss)
        mlflow.log_metric("f1_macro", val_f1)

        # Guardar modelo temporal (opcional)
        signature = infer_signature(X_val, y_pred)
        mlflow.sklearn.log_model(model, "model", input_example=X_val[:5], signature=signature)

    # Optuna maximiza el F1 macro
    return val_f1

#### Flujo de búsqueda

In [11]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
study_lr = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
with mlflow.start_run(run_name="Logistic Regression Optimization (Optuna)", nested=True):
    study_lr.optimize(objective_logreg, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params_lr = study_lr.best_params

    mlflow.log_params(best_params_lr)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "Stress Level Prediction",
        "optimizer_engine": "optuna",
        "model_family": "logistic_regression",
        "feature_set_version": 1,
    })

    mlflow.sklearn.autolog(log_models=False)

    # Entrenar modelo final con mejores hiperparámetros
    final_model_lr = LogisticRegression(**best_params_lr, n_jobs=-1, random_state=42)
    final_model_lr.fit(X_train_bal, y_train_bal)
    y_pred = final_model_lr.predict(X_val)
    y_proba = final_model_lr.predict_proba(X_val)
    y_pred = final_model_lr.predict(X_val)

    val_logloss = log_loss(y_val, y_proba)
    val_acc = accuracy_score(y_val, y_pred)
    val_f1 = f1_score(y_val, y_pred, average="macro")
    mlflow.log_metric("f1", val_f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    feature_names_final = top_features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])


    mlflow.sklearn.log_model(final_model_lr, "model", input_example=input_example, signature=signature)


[I 2025-11-23 16:45:47,555] A new study created in memory with name: no-name-9d3279d6-3144-412c-99a6-01e9975d561d
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/23 16:45:58 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 16:46:17 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 16:46:20 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environ

🏃 View run silent-ray-141 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/bee7f338d8ae4a01bdbb15d00cf74fa2
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


[I 2025-11-23 16:46:27,032] Trial 0 finished with value: 0.6755882039816886 and parameters: {'penalty': 'l2', 'C': 0.39079671568228835, 'class_weight': None}. Best is trial 0 with value: 0.6755882039816886.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/23 16:46:37 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 16:46:51 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 16:46:52 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please

🏃 View run upset-boar-731 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/3847cdce2d48401c9f0fc4140f0d393d
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


[I 2025-11-23 16:46:59,137] Trial 1 finished with value: 0.6774861114072793 and parameters: {'penalty': 'l2', 'C': 1.7718847354806828, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.6774861114072793.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
2025/11/23 16:47:04 WARNING mlflow.models.model: `arti

...

🏃 View run ambitious-hog-297 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/08acc42eb215477f97d69815b9c1bea2
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


[I 2025-11-23 16:50:44,992] Trial 8 finished with value: 0.6774861114072793 and parameters: {'penalty': 'l1', 'C': 23.386439256208728, 'class_weight': 'balanced'}. Best is trial 1 with value: 0.6774861114072793.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1272: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.8. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
2025/11/23 16:50:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 16:51:03 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 16:51:04 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. P

🏃 View run painted-mule-227 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/51394257539c4eb8919c38247c2b60be
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 16:51:26 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 16:51:38 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but LogisticRegression was fitted without feature names
  warnings.warn(
2025/11/23 16:51:38 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run Logistic Regression Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/21e47a39c7e447068be9d304d20bc37d
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


### Random Forest
---

El segundo modelo que se entrena es un Random Forest. Al igual que con la regresión logística, se utiliza Optuna para la optimización de hiperparámetros y MLflow para el seguimiento de experimentos.

Los hiperparámetros que se optimizan son:
- *N estimators*: Número de árboles en el bosque.
- *Max depth*: Profundidad máxima de los árboles.
- *Min samples split*: Número mínimo de muestras necesarias para dividir un nodo.
- *Min samples leaf*: Número mínimo de muestras necesarias en una hoja.
- *Max features*: Número de características a considerar al buscar la mejor división.
- *Class weight (None, balanced)*: Ajusta los pesos de las clases para manejar conjuntos de datos desequilibrados.

La función objetivo para Optuna entrena el modelo con los hiperparámetros sugeridos y evalúa su rendimiento utilizando la métrica F1 macro en el conjunto de validación. El objetivo es maximizar esta métrica al igual que con la regresión logística.

#### Función objetivo

In [16]:
def objective_rf(trial: optuna.trial.Trial):
    # Hiperparámetros a buscar
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 50, 500),
        "max_depth": trial.suggest_int("max_depth", 3, 30),
        "min_samples_split": trial.suggest_int("min_samples_split", 2, 20),
        "min_samples_leaf": trial.suggest_int("min_samples_leaf", 1, 20),
        "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2", None]),
        "class_weight": None,
        "random_state": 42,
        "n_jobs": -1
    }

    # Entrenamiento y evaluación
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "random_forest")
        mlflow.log_params(params)

        # Entrenar
        clf = RandomForestClassifier(**params)
        clf.fit(X_train_bal, y_train_bal)

        # Validación
        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        # Métricas
        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        # Logguear métricas
        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)

        # Guardar modelo del trial
        signature = infer_signature(X_val, y_pred)

        mlflow.sklearn.log_model(clf, "model", input_example=X_val[:5], signature=signature)

    return val_f1


#### Flujo de búsqueda

In [17]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
study_rf = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
with mlflow.start_run(run_name="RandomForest Hyperparameter Optimization (Optuna)", nested=True):
    study_rf.optimize(objective_rf, n_trials=10)
    best_rf = study_rf.best_params
    mlflow.log_params(best_rf)
   

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "Stress Level Predicition",
        "optimizer_engine": "optuna",
        "model_family": "random_forest",
        "feature_set_version": 1,
    })

    mlflow.sklearn.autolog(log_models=False)

    # Entrenar modelo final con mejores hiperparámetros
    final_model = RandomForestClassifier(**best_rf, n_jobs=-1, random_state=42)
    final_model.fit(X_train_bal, y_train_bal)
    y_val_proba = final_model.predict_proba(X_val)
    y_pred = final_model.predict(X_val)
    val_logloss = log_loss(y_val, y_val_proba)
    val_acc = accuracy_score(y_val, y_pred)
    val_f1 = f1_score(y_val, y_pred, average="macro")

    mlflow.log_metric("f1", val_f1)

    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    feature_names_final = top_features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])

    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)

[I 2025-11-23 17:13:09,065] A new study created in memory with name: no-name-cd764503-87a4-4618-b2dc-8e6185a12f35
2025/11/23 17:13:15 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:13:23 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 17:13:24 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-23 17:13:42,221] Trial 0 finished with value: 0.6841273641923257 and parameters: {'n_estimators': 218, 'max_depth': 29, 'min_samples_split': 15, 'min_samples_leaf': 12, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6841273641923257.


🏃 View run silent-lark-419 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/8f7efe4cf3f24654aa0bafd5b478e565
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:13:50 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:13:59 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 17:14:00 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-23 17:14:53,767] Trial 1 finished with value: 0.6762294241475609 and parameters: {'n_estimators': 440, 'max_depth': 19, 'min_samples_split': 15, 'min_samples_leaf': 1, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.6841273641923257.


🏃 View run resilient-eel-904 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/a26fbd87a8484642815df4b0d6bcd07f
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:14:59 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:15:06 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 17:15:07 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-23 17:15:12,199] Trial 2 finished with value: 0.6615846615846616 and parameters: {'n_estimators': 132, 'max_depth': 8, 'min_samples_split': 7, 'min_samples_leaf': 11, 'max_features': None}. Best is trial 0 with value: 0.6841273641923257.


...

🏃 View run abrasive-tern-684 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/81c649153a6642fa84479cf4bc1ab6eb
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:17:41 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:17:50 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
2025/11/23 17:17:50 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.
[I 2025-11-23 17:18:01,513] Trial 9 finished with value: 0.6617925223802001 and parameters: {'n_estimators': 210, 'max_depth': 10, 'min_samples_split': 12, 'min_samples_leaf': 3, 'max_features': None}. Best is trial 3 with value: 0.6934559474523425.


🏃 View run gregarious-eel-132 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/14cdc4fe1033431497acf67beaa8f94f
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:18:06 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:18:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\sklearn\utils\validation.py:2742: UserWarning: X has feature names, but RandomForestClassifier was fitted without feature names
  warnings.warn(
2025/11/23 17:18:15 INFO mlflow.models.model: Found the following environment variables used during model inference: [DATABRICKS_HOST, DATABRICKS_TOKEN]. Please check if you need to set them when deploying the model. To disable this message, set environment variable `MLFLOW_RECORD_ENV_VARS_IN_MODEL_LOGGING` to `false`.


🏃 View run RandomForest Hyperparameter Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/fccb623643c445f4b726aa58f7e34c22
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


### XGBoost
---

El tercer modelo que se entrena es un XGBoost. Al igual que con los modelos anteriores, se utiliza Optuna para la optimización de hiperparámetros y MLflow para el seguimiento de experimentos.

Los hiperparámetros que se optimizan son:
- *N estimators*: Número de árboles en el modelo.
- *Max depth*: Profundidad máxima de los árboles.
- *Learning rate*: Tasa de aprendizaje del modelo.
- *Subsample*: Proporción de muestras utilizadas para entrenar cada árbol.
- *Colsample bytree*: Proporción de características utilizadas para entrenar cada árbol.
- *Gamma*: Reducción mínima de la función de pérdida requerida para hacer una partición
- *Min child weight*: Peso mínimo de la suma de instancias necesarias en un nodo hijo.

La función objetivo para Optuna entrena el modelo con los hiperparámetros sugeridos y evalúa su rendimiento utilizando la métrica F1 macro en el conjunto de validación. El objetivo es maximizar esta métrica al igual que con los modelos anteriores.

#### Función objetivo

In [ ]:
def objective_xgb(trial: optuna.trial.Trial):
    # Hiperparámetros a buscar
    params = {
        "max_depth": trial.suggest_int("max_depth", 3, 16),
        "n_estimators": trial.suggest_int("n_estimators", 50, 800),
        "learning_rate": trial.suggest_float("learning_rate", 1e-3, 0.5, log=True),
        "gamma": trial.suggest_float("gamma", 0.0, 5.0),
        "min_child_weight": trial.suggest_float("min_child_weight", 1e-1, 50.0, log=True),
        "subsample": trial.suggest_float("subsample", 0.5, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.4, 1.0),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-8, 10.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-8, 10.0, log=True),
        "seed":42
    }

    # Entrenamiento y evaluación
    with mlflow.start_run(nested=True):
        mlflow.set_tag("model_family", "xgboost")
        # log params 
        mlflow.log_params(params)

        # Entrenar
        clf = xgb.XGBClassifier(**params)
        clf.fit(X_train_bal, y_train_bal, eval_set=[(X_val, y_val)], verbose=False)

        # Validación
        y_proba = clf.predict_proba(X_val)
        y_pred = clf.predict(X_val)

        val_logloss = log_loss(y_val, y_proba)
        val_acc = accuracy_score(y_val, y_pred)
        val_f1 = f1_score(y_val, y_pred, average="macro")

        # Log métricas
        mlflow.log_metric("val_log_loss", val_logloss)
        mlflow.log_metric("val_accuracy", val_acc)
        mlflow.log_metric("val_f1_macro", val_f1)

        signature = infer_signature(X_val, y_pred[:5])

        mlflow.xgboost.log_model(clf, artifact_path="model", input_example=X_val[:5], signature=signature)

    return val_f1

#### Flujo de búsqueda

In [ ]:
mlflow.sklearn.autolog(log_models=False)

# ------------------------------------------------------------
# Ejecutar la optimización (n_trials = número de intentos)
#    - Cada trial ejecuta la función objetivo con un set distinto de hiperparámetros.
#    - Abrimos un run "padre" para agrupar toda la búsqueda.
# ------------------------------------------------------------
study_xgb = optuna.create_study(direction="maximize", sampler=TPESampler(seed=42))
with mlflow.start_run(run_name="XGBoost Optimization (Optuna)", nested=True):
    study_xgb.optimize(objective_xgb, n_trials=10)

    # --------------------------------------------------------
    # Recuperar y registrar los mejores hiperparámetros
    # --------------------------------------------------------
    best_params_xgb = study_xgb.best_params

    mlflow.log_params(best_params_xgb)

    # Etiquetas del run "padre" (metadatos del experimento)
    mlflow.set_tags({
        "project": "Stress Level Predicition",
        "optimizer_engine": "optuna",
        "model_family": "xgboost",
        "feature_set_version": 1,
    })

    mlflow.sklearn.autolog(log_models=False)

    # Entrenar modelo final con mejores hiperparámetros
    final_model = xgb.XGBClassifier(**best_params_xgb, n_jobs=-1, random_state=42)
    final_model.fit(X_train_bal, y_train_bal)
    y_pred = final_model.predict(X_val)
    y_val_proba = final_model.predict_proba(X_val)
    xgb_val_logloss = log_loss(y_val, y_val_proba)
    xgb_val_acc = accuracy_score(y_val, y_pred)
    xgb_val_f1 = f1_score(y_val, y_pred, average="macro")

    mlflow.log_metric("f1", val_f1)
    
    pathlib.Path("preprocessor").mkdir(exist_ok=True)
    with open("preprocessor/preprocessor.b", "wb") as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact("preprocessor/preprocessor.b", artifact_path="preprocessor")

    feature_names_final = top_features
    input_example = pd.DataFrame(X_val[:5], columns=feature_names_final)
    signature = infer_signature(input_example, y_val[:5])

    mlflow.sklearn.log_model(final_model, "model", input_example=input_example, signature=signature)


[I 2025-11-23 17:04:54,511] A new study created in memory with name: no-name-65ee507d-0946-427f-b85a-60d8e202e706
2025/11/23 17:05:03 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [17:05:05] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/23 17:05:20 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [17:05:20] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_ap

🏃 View run adaptable-hare-803 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/9c7bf48bc0774941933499537cdbe22a
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:05:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [17:05:45] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/23 17:05:57 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [17:05:57] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)
[I 

🏃 View run rare-donkey-381 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/ce9b4980ba2b49ab850466a5cb76a978
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:06:16 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [17:06:17] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/23 17:06:29 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [17:06:29] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)
[I 

...

🏃 View run classy-jay-230 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/a5077322e3484546ae7cc6c590b38984
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:09:52 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1115: UserWarning: [17:09:53] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1575: Saving model in the UBJSON format as default.  You can use a file extension: `json` or `ubj` to choose between formats.
  self.get_booster().save_model(fname)
2025/11/23 17:10:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
c:\Users\cesar\Documents\5TO SEMESTRE ITESO\PROYECTO CIENCIA DATOS\project1-pcd\.venv\Lib\site-packages\xgboost\sklearn.py:1124: UserWarning: [17:10:00] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api.cc:1511: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  self.get_booster().load_model(fname)
[I 

🏃 View run traveling-carp-742 at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/8de42d19dd8f4f1d9d9583d5e0b2f39f
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


2025/11/23 17:10:19 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/23 17:10:26 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run XGBoost Optimization (Optuna) at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794/runs/38f5ae1f49c547bb9ee21b2ad5e5b6d9
🧪 View experiment at: https://dbc-fe5c4f3a-9d0d.cloud.databricks.com/ml/experiments/917681686631794


### Registrar modelo Champion
---

In [18]:
model_name = "workspace.default.equipo1-proyecto"

In [19]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1 DESC"],
    output_format="list"
)

#Obtener el mejor run
if len(runs) > 0:
   best_run = runs[0]
   print("🏆 Champion Run encontrado:")
   print(f"Run ID: {best_run.info.run_id}")
   print(f"f1-score: {best_run.data.metrics.get('f1')}")
   print(f"Params: {best_run.data.params}")
else:
   print("⚠️ No se encontraron runs con métrica f1-score.")

🏆 Champion Run encontrado:
Run ID: fccb623643c445f4b726aa58f7e34c22
f1-score: 0.6934559474523425
Params: {'bootstrap': 'True', 'ccp_alpha': '0.0', 'class_weight': 'None', 'criterion': 'gini', 'max_depth': '11', 'max_features': 'sqrt', 'max_leaf_nodes': 'None', 'max_samples': 'None', 'min_impurity_decrease': '0.0', 'min_samples_leaf': '10', 'min_samples_split': '8', 'min_weight_fraction_leaf': '0.0', 'monotonic_cst': 'None', 'n_estimators': '112', 'n_jobs': '-1', 'oob_score': 'False', 'random_state': '42', 'verbose': '0', 'warm_start': 'False'}


In [20]:
run_id = best_run.info.run_id

In [21]:
result = mlflow.register_model(
    model_uri=f"runs:/{best_run.info.run_id}/model",
    name=model_name
)

Registered model 'workspace.default.equipo1-proyecto' already exists. Creating a new version of this model...
2025/11/23 17:19:58 WARNING mlflow.tracking._model_registry.fluent: Run with id fccb623643c445f4b726aa58f7e34c22 has no artifacts at artifact path 'model', registering model based on models:/m-ce7c34ccfcf444c48f6c8167875911c3 instead
Uploading artifacts: 100%|██████████| 9/9 [00:05<00:00,  1.64it/s]
Created version '7' of model 'workspace.default.equipo1-proyecto'.


In [22]:
client = MlflowClient()

model_version = result.version
new_alias = "Champion"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

date = datetime.today()

client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1763940007385, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='The model version 7 was transitioned to Champion on 2025-11-23 17:20:14.881349', last_updated_timestamp=1763940015129, metrics=[<Metric: dataset_digest='', dataset_name='', key='f1', model_id='m-ce7c34ccfcf444c48f6c8167875911c3', run_id='fccb623643c445f4b726aa58f7e34c22', step=0, timestamp=1763939886251, value=0.6934559474523425>,
 <Metric: dataset_digest='', dataset_name='', key='training_accuracy_score', model_id='m-ce7c34ccfcf444c48f6c8167875911c3', run_id='fccb623643c445f4b726aa58f7e34c22', step=0, timestamp=1763939883206, value=0.9213675213675213>,
 <Metric: dataset_digest='', dataset_name='', key='training_f1_score', model_id='m-ce7c34ccfcf444c48f6c8167875911c3', run_id='fccb623643c445

### Registrar modelo Challenger
---

In [23]:
runs = mlflow.search_runs(
    experiment_names=[EXPERIMENT_NAME],
    order_by=["metrics.f1 DESC"],
    output_format="list"
)

# Obtener el segundo mejor (challenger)
if len(runs) > 1:
    challenger_run = runs[1]
    print("Challenger Run encontrado:")
    print(f"Run ID: {challenger_run.info.run_id}")
    print(f"f1-score: {challenger_run.data.metrics.get('f1')}")
    print(f"Params: {challenger_run.data.params}")


Challenger Run encontrado:
Run ID: 38f5ae1f49c547bb9ee21b2ad5e5b6d9
f1-score: 0.6893100934455564
Params: {'colsample_bytree': '0.4837849622489775', 'gamma': '0.05739466880087418', 'learning_rate': '0.005137104257005199', 'max_depth': '12', 'min_child_weight': '12.06981395637106', 'n_estimators': '568', 'reg_alpha': '0.00159243131800919', 'reg_lambda': '1.1643095197482896e-05', 'subsample': '0.9080711631444247'}


In [24]:
run_id = challenger_run.info.run_id

In [25]:
result = mlflow.register_model(
    model_uri=f"runs:/{challenger_run.info.run_id}/model",
    name=model_name
)

Registered model 'workspace.default.equipo1-proyecto' already exists. Creating a new version of this model...
2025/11/23 17:20:20 WARNING mlflow.tracking._model_registry.fluent: Run with id 38f5ae1f49c547bb9ee21b2ad5e5b6d9 has no artifacts at artifact path 'model', registering model based on models:/m-641b7493e57640c995292ed98bcd50b7 instead
Uploading artifacts: 100%|██████████| 8/8 [00:04<00:00,  1.67it/s]
Created version '8' of model 'workspace.default.equipo1-proyecto'.


In [26]:
client = MlflowClient()

model_version = result.version
new_alias = "Challenger"

client.set_registered_model_alias(
    name=model_name,
    alias=new_alias,
    version=result.version
)

date = datetime.today()

client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_alias} on {date}"
)

<ModelVersion: aliases=[], creation_timestamp=1763940031368, current_stage=None, deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description=('The model version 8 was transitioned to Challenger on 2025-11-23 '
 '17:20:37.903810'), last_updated_timestamp=1763940038149, metrics=[<Metric: dataset_digest='', dataset_name='', key='f1', model_id='m-641b7493e57640c995292ed98bcd50b7', run_id='38f5ae1f49c547bb9ee21b2ad5e5b6d9', step=0, timestamp=1763939417938, value=0.6893100934455564>], model_id='m-641b7493e57640c995292ed98bcd50b7', name='workspace.default.equipo1-proyecto', params=[<LoggedModelParameter: key='colsample_bytree', value='0.4837849622489775'>,
 <LoggedModelParameter: key='min_child_weight', value='12.06981395637106'>,
 <LoggedModelParameter: key='n_estimators', value='568'>,
 <LoggedModelParameter: key='gamma', value='0.0573

El mejor modelo, champion, fue `Logistic Regression` con un *f1-score* de *0.3529*, superando ligeramente al challenger, un modelo `XGBoost` con un *f1-score* de *0.3508*.
El criterio de evaluación utilizado fue el **f1-score**, adecuado para este caso debido al desbalance en la variable objetivo, ya que considera tanto la precisión como el recall para medir el desempeño general del modelo.
